# 🚀 FATFORMER-XLA: ĐÁNH GIÁ KIỂM CHUẨN NGOẠI VI COMMUNITYFORENSICS-EVAL (CVPR 2025)
### ĐO ĐẠC ĐA CHIỀU: 3 MÔ HÌNH × 2 ĐIỀU KIỆN (CLEAN & JPEG Q=30)

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mục tiêu**: Đánh giá năng lực tổng quát hóa Zero-shot và độ bền vững pháp y trên 2.000 ảnh Unseen của bộ kiểm chuẩn ngoại vi độc lập **CommunityForensics-Eval (CVPR 2025)**.  
> **3 Mô hình đối chứng**:  
> 1. **Baseline CVPR 2024** (Mô hình gốc không có SRM/Gating).  
> 2. **WiSE-FT (Mode 1)**: Trọng số nội suy đa miền cân bằng hiệu năng ảnh sạch.  
> 3. **Anchored Recovery (Mode 2)**: Trọng số phục hồi neo chặn chuyên biệt cho biến dạng nén sâu mạng xã hội.  
> **2 Điều kiện kiểm thử**:  
> - **Bảng A**: Ảnh sạch phòng lab (Clean).  
> - **Bảng B**: Ảnh nén sâu mạng xã hội (In-memory JPEG Quality=30).  
> **Hạ tầng kế thừa**: Checkpoint và dữ liệu `.tar` lưu trữ tại `MyDrive/Fatformer` (Google Drive 5TB).

### Bước 1: Kết nối Google Drive & Kiểm tra Tài nguyên Phần cứng GPU

In [1]:
from google.colab import drive
import os, sys, torch

print("[*] Đang liên kết Google Drive 5TB...")
drive.mount('/content/drive')

print(f"[*] Phiên bản PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[✓] Thiết bị tính toán: GPU {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    print("[!] CẢNH BÁO: Không tìm thấy GPU! Đang chạy trên CPU (tốc độ suy luận sẽ chậm hơn).")

!df -h /content | tail -1

[*] Đang liên kết Google Drive 5TB...
Mounted at /content/drive
[*] Phiên bản PyTorch: 2.11.0+cu130
[✓] Thiết bị tính toán: GPU Tesla T4 (14.56 GB VRAM)
overlay         236G   43G  194G  19% /


### Bước 2: Đồng Bộ Mã Nguồn Mới Nhất Từ GitHub (Nhánh `hoang`)

In [2]:
REPO_DIR = "/content/fatformer-xla"
REPO_URL = "https://github.com/LeeVHoangtk3/fatformer-xla.git"
BRANCH = "hoang"

if not os.path.exists(REPO_DIR):
    print(f"[*] Đang clone mã nguồn từ GitHub (nhánh {BRANCH}): {REPO_URL}...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"[*] Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật commit mới nhất (git pull)...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

[*] Đang clone mã nguồn từ GitHub (nhánh hoang): https://github.com/LeeVHoangtk3/fatformer-xla.git...
Cloning into '/content/fatformer-xla'...
remote: Enumerating objects: 1615, done.
remote: Counting objects: 100% (435/435), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 1615 (delta 262), reused 330 (delta 181), pack-reused 1180 (from 1)
Receiving objects: 100% (1615/1615), 11.49 MiB | 40.85 MiB/s, done.
Resolving deltas: 100% (899/899), done.
/content/fatformer-xla
[✓] Thư mục làm việc hiện tại: /content/fatformer-xla


### Bước 3: Giải Nén Dữ Liệu `test_communityforensics.tar` Sang NVMe SSD Colab
Giải nén từ Google Drive sang SSD Colab `/content/dataset_local/` để đạt tốc độ nạp I/O cao nhất.

In [3]:
import os, tarfile

DRIVE_TAR_PATH = "/content/drive/MyDrive/Fatformer/datasets/test_communityforensics.tar"
LOCAL_DIR = "/content/dataset_local/test_communityforensics"

# Kiểm tra nếu thư mục cục bộ đã có đủ 5 tập con thì không cần giải nén lại
need_extract = True
if os.path.exists(LOCAL_DIR):
    subsets = [d for d in os.listdir(LOCAL_DIR) if os.path.isdir(os.path.join(LOCAL_DIR, d))]
    if len(subsets) >= 5:
        print(f"[✓] Phát hiện dữ liệu 5 tập con đã có sẵn tại {LOCAL_DIR}. Bỏ qua bước giải nén!")
        need_extract = False

if need_extract:
    if not os.path.exists(DRIVE_TAR_PATH):
        raise FileNotFoundError(f"[!] Không tìm thấy tệp {DRIVE_TAR_PATH} trên Google Drive. Hãy chạy notebook chuẩn bị dữ liệu trước!")

    tar_sz_gb = os.path.getsize(DRIVE_TAR_PATH) / (1024 ** 3)
    print(f"[*] Đang giải nén {DRIVE_TAR_PATH} ({tar_sz_gb:.2f} GB) sang SSD Colab...")
    os.makedirs("/content/dataset_local", exist_ok=True)
    with tarfile.open(DRIVE_TAR_PATH, "r") as tar:
        tar.extractall("/content/dataset_local")
    print("[✓] Giải nén thành công toàn bộ 2.000 ảnh chuẩn hóa!")

# Kiểm tra số lượng ảnh
!find /content/dataset_local/test_communityforensics -type f -name "*.png" | wc -l

[*] Đang giải nén /content/drive/MyDrive/Fatformer/datasets/test_communityforensics.tar (4.86 GB) sang SSD Colab...


/tmp/ipykernel_953/3497914844.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content/dataset_local")


[✓] Giải nén thành công toàn bộ 2.000 ảnh chuẩn hóa!
2000


### Bước 4: Kích Hoạt Kiểm Chuẩn Tự Động 3 Mô Hình Trên 2 Điều Kiện
Đo đạc Macro ACC, Real ACC, Fake ACC, ROC-AUC và AP trên cả 5 họ mô hình bộ sinh.

In [4]:
!python tools/eval_communityforensics.py \
    --data_dir /content/dataset_local/test_communityforensics \
    --output_dir /content/drive/MyDrive/Fatformer/logs/cfe_benchmark

FATFORMER-XLA: PIPELINE KIỂM CHUẨN NGOẠI VI COMMUNITYFORENSICS-EVAL (CVPR 2025)
• Thiết bị tính toán: CUDA (Tesla T4)
• Thư mục dữ liệu:   /content/dataset_local/test_communityforensics

[*] Trạng thái định vị Checkpoints:
  - Baseline CVPR 2024: /content/drive/MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth
  - WiSE-FT (Mode 1): /content/drive/MyDrive/Fatformer/checkpoint/fatformer_wise_ft_best.pth
  - Anchored Recovery (Mode 2): /content/drive/MyDrive/Fatformer/checkpoint/fatformer_anchored_recovery_best.pth

[*] Đã quét được 2000 ảnh từ 5 tập con:
  • 1_commercial             :  600 ảnh (300 Real / 300 Fake)
  • 2_latent_diffusion       :  600 ảnh (300 Real / 300 Fake)
  • 3_pixel_diffusion        :  400 ảnh (200 Real / 200 Fake)
  • 4_modern_gans            :  200 ảnh (100 Real / 100 Fake)
  • 5_other_autoregressive   :  200 ảnh (100 Real / 100 Fake)

BẮT ĐẦU ĐÁNH GIÁ CHUỖI: ẢNH SẠCH (CLEAN)

[*] Đang đánh giá mô hình [Baseline CVPR 2024] trên ẢNH SẠCH (CLEAN)...
[*] Dang kh

### Bước 5: Xem Trực Quan Báo Cáo Nghiệm Thu Markdown
Hiển thị bảng ma trận Bảng A (Clean) và Bảng B (JPEG Q=30) trực tiếp trong Notebook.

In [5]:
from IPython.display import display, Markdown

REPORT_MD = "/content/drive/MyDrive/Fatformer/logs/cfe_benchmark/communityforensics_benchmark_report.md"
if os.path.exists(REPORT_MD):
    with open(REPORT_MD, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print(f"[*] Báo cáo sẽ xuất hiện tại: {REPORT_MD}")

# BÁO CÁO KẾT QUẢ KIỂM ĐỊNH NGOẠI VI: COMMUNITYFORENSICS-EVAL (CVPR 2025)
> **Dự án**: FatFormer-XLA | **Thời gian thực hiện**: 2026-10-26 18:59:15
> **Tổng số mẫu**: 2.000 ảnh (1.000 Real / 1.000 Fake trên 5 họ kiến trúc)

---

### Bảng A: Kết Quả Kiểm Chuẩn Trên Miền Ảnh Sạch (Clean CommunityForensics-Eval — 2.000 Ảnh)
*Mục đích: Chứng minh WiSE-FT bảo toàn độ chính xác đỉnh cao và chống báo oan ảnh thật trên các mô hình 2024–2025.*

| Tập Kiểm Chuẩn Con | Họ Kiến Trúc & Mô Hình Đại Diện | Số Mẫu | Baseline CVPR 2024 ($T^*=0.50$) | WiSE-FT Mode 1 ($T=1.08, T^*=0.50$) | Anchored Mode 2 ($T=1.28, T^*=0.40$) |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **1_commercial** | Commercial (MJ v6, DALL-E 3) | 600 | 56.17% | **50.00%** | 73.17% |
| **2_latent_diffusion** | LatDiff (SDXL, Flux.1) | 600 | 64.50% | **50.00%** | 72.00% |
| **3_pixel_diffusion** | PixDiff (DeepFloyd, ADM) | 400 | 65.00% | **50.00%** | 62.75% |
| **4_modern_gans** | GANs (StyleGAN3, GigaGAN) | 200 | 95.00% | **70.50%** | 71.50% |
| **5_other_autoregressive** | Other / Autoregressive (Muse, LlamaGen) | 200 | 59.50% | **50.00%** | 60.50% |
| 🏆 **SUBSET-MACRO MEAN** | **Trung bình không trọng số (5 họ)** | 2.000 | 68.03% | **54.10% 🏆** | 67.98% |
| 📊 **OVERALL POOLED MICRO** | **Toàn bộ 2.000 ảnh gộp (Micro ACC)** | 2.000 | 64.65% | **52.05% 🏆** | 69.30% |
| • Clean Real ACC | COCO, FFHQ, RAISE, LAION | 1.000 | 94.00% | **100.00% 🏆** | 48.10% |
| • Clean Fake ACC | Toàn bộ 5 họ AI mới | 1.000 | 35.30% | 4.10% | **90.50% 🔥** |
| • Clean ROC-AUC | Độc lập ngưỡng cắt $T^*$ | 2.000 | 76.21% | **57.07% 🏆** | 79.28% |
| • Clean Average Precision (AP) | Năng lực phân tách | 2.000 | 78.22% | **60.41% 🏆** | 73.53% |

---

### Bảng B: Kết Quả Kiểm Chuẩn Dưới Biến Dạng Nén Sâu (JPEG Q=30 CommunityForensics-Eval — 2.000 Ảnh)
*Mục đích: Chứng minh Baseline gốc và WiSE-FT sụp đổ khi ảnh AI mới bị nén qua mạng xã hội, trong khi Anchored Recovery cứu vãn ngoạn mục.*

| Chỉ Số Đo Đạc Dưới Nén Sâu ($Q=30$) | Số Mẫu | Baseline CVPR 2024 | WiSE-FT (Chế Độ 1) | Anchored Recovery (Chế Độ 2) |
| :--- | :---: | :---: | :---: | :---: |
| • **1_commercial Fake ACC** (Commercial (MJ v6, DALL-E 3)) | 300 Fake | 0.00% | 0.00% | **66.67% 🔥** |
| • **2_latent_diffusion Fake ACC** (LatDiff (SDXL, Flux.1)) | 300 Fake | 0.00% | 0.00% | **37.67% 🔥** |
| • **3_pixel_diffusion Fake ACC** (PixDiff (DeepFloyd, ADM)) | 200 Fake | 0.50% | 0.00% | **20.00% 🔥** |
| • **4_modern_gans Fake ACC** (GANs (StyleGAN3, GigaGAN)) | 100 Fake | 8.00% | 2.00% | **19.00% 🔥** |
| • **5_other_autoregressive Fake ACC** (Other / Autoregressive (Muse, LlamaGen)) | 100 Fake | 0.00% | 0.00% | **36.00% 🔥** |
| 🏆 **SUBSET-MACRO FAKE ACC** | **Trung bình không trọng số Fake ACC (5 họ)** | 1.000 Fake | 1.70% | 0.40% | **35.87% 🏆** |
| 🔥 **OVERALL POOLED FAKE ACC** | **Toàn bộ 1.000 ảnh Fake gộp (Micro)** | 1.000 Fake | **0.90%** | **0.20%** | **40.80% 🏆** |
| • Overall Q=30 Macro ACC | Toàn bộ 2.000 ảnh gộp | 2.000 | 50.35% | 50.10% | **49.60%** |
| • Overall Q=30 Real ACC | Bảo vệ ảnh thật dưới nén sâu | 1.000 | 99.80% | 100.00% | 58.40% |
| • Overall Q=30 ROC-AUC | Độc lập ngưỡng cắt $T^*$ | 2.000 | 50.00% | 50.93% | **50.07% 🏆** |
| • Overall Q=30 AP | Năng lực phân tách dưới nén sâu | 2.000 | 55.10% | 54.11% | **50.57% 🏆** |

---
> **Kết luận tổng quát**: Thí nghiệm trên CommunityForensics-Eval (CVPR 2025) khẳng định tính thiết yếu của Kiến trúc Hệ thống Kép: WiSE-FT tối ưu cho môi trường phòng lab / ảnh chất lượng cao, còn Anchored Recovery là vũ khí sống còn khi ảnh AI bị phát tán qua các kênh mạng xã hội nén sâu.